In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


In [3]:
# Load the same Netflix dataset you already used
df = pd.read_csv("Data/NETFLIX MOVIES AND TV SHOWS CLUSTERING.csv")

# Quick check
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...


In [4]:
# Merge relevant fields into one text string per movie/show
df["document"] = (
    df["title"].fillna("") + " " +
    df["listed_in"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["cast"].fillna("") + " " +
    df["rating"].fillna("")
)
df["document"].head()


0    3% International TV Shows, TV Dramas, TV Sci-F...
1    7:19 Dramas, International Movies After a deva...
2    23:59 Horror Movies, International Movies When...
3    9 Action & Adventure, Independent Movies, Sci-...
4    21 Dramas A brilliant group of students become...
Name: document, dtype: object

In [5]:
# Load a pretrained sentence transformer
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all documents into embeddings (vectors)
embeddings = model.encode(df["document"].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings)

embeddings.shape  # should be (num_movies, 384)


Batches:   0%|          | 0/244 [00:00<?, ?it/s]

(7787, 384)

In [6]:
# Create FAISS index for fast similarity search
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"Indexed {index.ntotal} movies/shows.")


Indexed 7787 movies/shows.


In [7]:
def recommend(title, top_k=10):
    try:
        idx = df[df['title'].str.lower() == title.lower()].index[0]
    except IndexError:
        return f"❌ '{title}' not found in dataset."
    
    vector = embeddings[idx:idx+1]
    distances, indices = index.search(vector, top_k+1)  # +1 includes itself
    
    results = []
    for i in indices[0][1:]:  # skip itself
        results.append(df.iloc[i]["title"])
    return results


In [8]:
print(recommend("Breaking Bad", 10))
print(recommend("Narcos", 10))
print(recommend("The Crown", 10))


['The Break', 'Reckoning', 'The Show', 'Medical Police', 'Somewhere Between', 'How to Fix a Drug Scandal', 'American Vandal', 'Criminal Minds', 'How to Get Away with Murder', 'You']
['Narcos: Mexico', 'El Cartel', 'El Cartel 2', 'Undercover Law', 'Pablo Escobar, el patrón del mal', 'Wild District', 'Drug Squad: Costa del Sol', 'Los tiempos de Pablo Escobar', 'Yankee', 'El señor de los Cielos']
['Reign', 'The Windsors', 'The Royal House of Windsor', 'The English Game', 'W1A', 'The Tudors', 'Kiss Me First', 'Dynasty', 'Hinterland', 'Vexed']


In [13]:
print(recommend("Dil Chahta Hai", 10))

['Dil', 'Phir Bhi Dil Hai Hindustani', 'Dil Hai Tumhaara', 'Dilwale', 'Dil Dhadakne Do', 'Hamara Dil Aapke Paas Hai', 'Dil Se', 'Ek Main Aur Ekk Tu', 'Dilan 1990', 'Rajma Chawal']
